# Lab 7d — Measuring $M_L$, $m_b$, and $M_S$ from Seismograms

<a target="_blank" href="https://colab.research.google.com/github/uw-geophysics-edu/ess-412-512-intro2seismology/blob/main/notebooks/07d_Magnitude_Measurement_Practice.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Learning Objectives

After completing this notebook you will be able to:

1. **Download** broadband seismograms from FDSN web services and remove the instrument response.
2. **Measure $M_L$** by simulating a Wood–Anderson seismometer and reading peak horizontal amplitudes.
3. **Measure $m_b$** from the maximum P-wave displacement/period ratio on a short-period vertical channel.
4. **Measure $M_S$** from the Rayleigh-wave amplitude on a long-period vertical channel.
5. **Compare** the three estimates with the catalog value and discuss why they may differ.

In [ ]:
# Install dependencies (Colab / first run)
# %pip install obspy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.taup import TauPyModel

client = Client("IRIS")
model  = TauPyModel(model="iasp91")

---
## 1. Choose an Earthquake

We search for a recent, **shallow**, moderate earthquake ($M_w$ 6–7) so that all three magnitude scales return meaningful values.  Feel free to adjust the time window or magnitude range.

In [ ]:
from obspy import UTCDateTime

t_end   = UTCDateTime.now()
t_start = t_end - 365 * 86400  # past year

cat = client.get_events(
    starttime=t_start, endtime=t_end,
    minmagnitude=6.0, maxmagnitude=7.0,
    maxdepth=40,           # km – keep it shallow for clear Rayleigh waves
    orderby="magnitude",
    limit=5,
)
print(cat)
event = cat[0]
origin = event.preferred_origin() or event.origins[0]
ev_lat, ev_lon, ev_dep_km = origin.latitude, origin.longitude, origin.depth / 1e3
ev_time = origin.time
catalog_mag = (event.preferred_magnitude() or event.magnitudes[0]).mag
print(f"\nSelected event: {ev_time}  Mw {catalog_mag:.1f}  "
      f"depth {ev_dep_km:.1f} km  ({ev_lat:.2f}, {ev_lon:.2f})")

---
## 2. Find a Broadband Station

We look for a BH-channel station at **teleseismic distance** (30°–80°) so that P, S, and Rayleigh waves are all well separated.

In [ ]:
from obspy.geodetics import locations2degrees

inv = client.get_stations(
    starttime=ev_time, endtime=ev_time + 3600,
    channel="BH?",
    minlatitude=ev_lat - 60, maxlatitude=ev_lat + 60,
    minlongitude=ev_lon - 60, maxlongitude=ev_lon + 60,
    level="channel",
)

# Pick the first station between 30° and 80° that has BHZ + BHN + BHE (or BH1/BH2)
chosen = None
for net in inv:
    for sta in net:
        dist_deg = locations2degrees(ev_lat, ev_lon, sta.latitude, sta.longitude)
        if 30 <= dist_deg <= 80:
            chans = {ch.code for ch in sta}
            if "BHZ" in chans and len(chans & {"BHN", "BH1"}) > 0:
                chosen = (net.code, sta.code, sta.latitude, sta.longitude, dist_deg)
                break
    if chosen:
        break

if chosen is None:
    raise RuntimeError("No suitable station found – try widening the search box.")

net_code, sta_code, sta_lat, sta_lon, dist_deg = chosen
print(f"Station: {net_code}.{sta_code}  Δ = {dist_deg:.1f}°  "
      f"({sta_lat:.2f}, {sta_lon:.2f})")

---
## 3. Predicted Arrivals (TauP)

We use the **iasp91** model to predict P and S arrival times.  These windows will be used to isolate phases for $m_b$ and $M_S$.

In [ ]:
arrivals = model.get_travel_times(
    source_depth_in_km=ev_dep_km,
    distance_in_degree=dist_deg,
    phase_list=["P", "S"],
)

t_P = None
t_S = None
for arr in arrivals:
    if arr.name == "P" and t_P is None:
        t_P = arr.time
    if arr.name == "S" and t_S is None:
        t_S = arr.time

print(f"Predicted P arrival: {t_P:.1f} s after origin")
print(f"Predicted S arrival: {t_S:.1f} s after origin")

---
## 4. Download Waveforms & Remove Instrument Response

We download **velocity** seismograms, then remove the instrument response to obtain **displacement** (needed for $m_b$ and $M_S$).  A separate copy is kept as velocity for the Wood–Anderson simulation.

In [ ]:
# Time window: 60 s before P to 200 s after predicted Rayleigh arrival
rayleigh_est = (dist_deg * 111.19) / 3.5  # rough group-velocity estimate (s)
t1 = ev_time + t_P - 60
t2 = ev_time + max(t_S + 300, rayleigh_est + 200)

st = client.get_waveforms(
    net_code, sta_code, "*", "BH?",
    starttime=t1, endtime=t2,
    attach_response=True,
)
st.merge(fill_value="interpolate")
print(st)

In [ ]:
# Displacement copy (for mb, Ms)
st_disp = st.copy()
st_disp.remove_response(output="DISP", pre_filt=[0.005, 0.01, 30, 35])

# Velocity copy (for ML Wood–Anderson simulation)
st_vel = st.copy()
st_vel.remove_response(output="VEL", pre_filt=[0.005, 0.01, 30, 35])

print("Displacement traces:", st_disp)
print("Velocity traces:", st_vel)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, tr in zip(axes, st_disp):
    t = tr.times(reftime=ev_time)
    ax.plot(t, tr.data * 1e6, "k", lw=0.5)  # µm
    ax.set_ylabel(f"{tr.id}\n(µm)")
    ax.axvline(t_P, color="blue", ls="--", lw=0.8, label="P")
    ax.axvline(t_S, color="red", ls="--", lw=0.8, label="S")
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("Time after origin (s)")
axes[0].set_title("Displacement seismograms")
plt.tight_layout()
plt.show()

---
## Exercise A — Local Magnitude $M_L$

The **local magnitude** was originally defined by Richter (1935) for Southern California using a **Wood–Anderson torsion seismometer** (natural period 0.8 s, damping 0.8, magnification 2800).

$$
M_L = \log_{10} A_{\text{WA}} + \text{distance correction}
$$

where $A_{\text{WA}}$ is the peak trace amplitude in **mm** on a Wood–Anderson instrument.

### Steps
1. Simulate a Wood–Anderson response on each horizontal component.
2. Bandpass filter 1–10 Hz.
3. Read the **peak zero-to-peak amplitude** on each horizontal.
4. Apply the **Hutton & Boore (1987)** distance correction.
5. Average the two horizontals.

> **Note:** $M_L$ is designed for *local/regional* distances (Δ < 6°).  At teleseismic range it is not standard, but we compute it here for comparison.

In [ ]:
from obspy.signal.invsim import estimate_wood_anderson_amplitude_using_response

# Wood–Anderson instrument constants (IASPEI standard)
WA_PAZ = {
    "poles": [(-6.2832 - 4.7124j), (-6.2832 + 4.7124j)],
    "zeros": [0j, 0j],
    "gain": 1.0,
    "sensitivity": 2800.0,
}

# Work on velocity traces (simulate_seismometer expects velocity input)
st_wa = st_vel.copy()
st_wa.filter("bandpass", freqmin=1.0, freqmax=10.0, corners=4, zerophase=True)
st_wa.simulate(paz_remove=None, paz_simulate=WA_PAZ)

# Peak amplitude on each horizontal (mm)
horizontals = [tr for tr in st_wa if tr.stats.channel[-1] in ("N", "E", "1", "2")]
amps_mm = []
for tr in horizontals:
    amp = np.max(np.abs(tr.data)) * 1e3  # m → mm
    amps_mm.append(amp)
    print(f"  {tr.id}: peak WA amplitude = {amp:.4f} mm")

A_wa = np.mean(amps_mm)
print(f"\nMean horizontal WA amplitude: {A_wa:.4f} mm")

In [ ]:
# Hutton & Boore (1987) distance correction for Southern California
from obspy.geodetics import degrees2kilometers

dist_km = degrees2kilometers(dist_deg)
print(f"Epicentral distance: {dist_km:.1f} km")

log_A0 = 1.110 * np.log10(dist_km / 100) + 0.00189 * (dist_km - 100) + 3.0
# ML = log10(A_wa) - log_A0   [A_wa in mm, Hutton-Boore convention]
# Note: Hutton-Boore tabulates -log(A0); we use the analytic form
ML = np.log10(A_wa) + log_A0
print(f"\n==> ML = {ML:.2f}")

---
## Exercise B — Body-wave Magnitude $m_b$

The **body-wave magnitude** (Gutenberg & Richter 1956) measures the P-wave amplitude on a short-period (~1 s) vertical seismogram:

$$
m_b = \log_{10}\!\left(\frac{A}{T}\right) + Q(\Delta, h)
$$

where $A$ is the ground-motion displacement amplitude in **µm**, $T$ is the dominant period in **s**, and $Q(\Delta, h)$ is the Veith & Clawson (1972) attenuation correction.

### Steps
1. Bandpass the vertical displacement trace to 0.5–2 Hz.
2. Window **5 s before to 30 s after the predicted P arrival**.
3. Measure the maximum peak-to-trough amplitude $A$ and corresponding period $T$.
4. Apply the $Q$ correction for the observed distance and focal depth.

In [ ]:
# Isolate BHZ displacement
tr_z = st_disp.select(channel="BHZ")[0].copy()
tr_z.filter("bandpass", freqmin=0.5, freqmax=2.0, corners=4, zerophase=True)

# P-wave window
p_abs = ev_time + t_P
win_start = p_abs - 5
win_end   = p_abs + 30
tr_p = tr_z.copy().trim(starttime=win_start, endtime=win_end)
t_sec = tr_p.times(reftime=p_abs)

# Plot P window
plt.figure(figsize=(10, 3))
plt.plot(t_sec, tr_p.data * 1e6, "k", lw=0.7)
plt.xlabel("Time relative to predicted P (s)")
plt.ylabel("Displacement (µm)")
plt.title("P-wave window (0.5–2 Hz vertical)")
plt.axvline(0, color="blue", ls="--", label="predicted P")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Measure A/T: find largest peak-to-trough swing
data = tr_p.data * 1e6  # µm
dt   = tr_p.stats.delta

# Simple approach: find max and adjacent min (or vice versa)
idx_max = np.argmax(data)
idx_min = np.argmin(data)

# Ensure they are adjacent half-cycles
if idx_max < idx_min:
    A_pp = data[idx_max] - data[idx_min]
    T = 2.0 * abs(idx_min - idx_max) * dt
else:
    A_pp = data[idx_min] - data[idx_max]
    T = 2.0 * abs(idx_max - idx_min) * dt
    A_pp = abs(A_pp)

A_half = A_pp / 2.0  # zero-to-peak amplitude

print(f"Peak-to-trough amplitude: {A_pp:.3f} µm")
print(f"Zero-to-peak amplitude A: {A_half:.3f} µm")
print(f"Dominant period T: {T:.2f} s")
print(f"log10(A/T): {np.log10(A_half / T):.3f}")

In [ ]:
# Veith & Clawson (1972) Q correction — simplified analytic approximation
# For shallow sources (h < 70 km) at teleseismic distances (20°–100°):
# Q ≈ 1.414 * log10(Δ) + 1.0  (fit to their Table 3 for h ≈ 33 km)

Q_vc = 1.414 * np.log10(dist_deg) + 1.0
print(f"Q(Δ={dist_deg:.1f}°, h={ev_dep_km:.0f} km) ≈ {Q_vc:.2f}")

mb = np.log10(A_half / T) + Q_vc
print(f"\n==> mb = {mb:.2f}")

---
## Exercise C — Surface-wave Magnitude $M_S$

The **surface-wave magnitude** uses the Rayleigh-wave amplitude near **20 s period** on a vertical seismogram:

$$
M_S = \log_{10}\!\left(\frac{A}{T}\right) + 1.66\,\log_{10}\Delta + 3.30
$$

This is the **Prague formula** (Vaněk et al. 1962), adopted by IASPEI.

### Steps
1. Bandpass the vertical displacement trace to periods 15–25 s (0.04–0.067 Hz).
2. Window the Rayleigh wave using group velocities 3.0–4.2 km/s.
3. Measure the maximum $A/T$.
4. Apply the Prague formula.

In [ ]:
tr_z2 = st_disp.select(channel="BHZ")[0].copy()
tr_z2.filter("bandpass", freqmin=0.04, freqmax=0.067, corners=4, zerophase=True)

# Rayleigh-wave window based on group velocity
dist_m  = dist_km * 1e3
t_fast  = ev_time + dist_m / 4200.0
t_slow  = ev_time + dist_m / 3000.0
tr_R = tr_z2.copy().trim(starttime=t_fast - 30, endtime=t_slow + 60)

t_sec_R = tr_R.times(reftime=ev_time)

plt.figure(figsize=(10, 3))
plt.plot(t_sec_R, tr_R.data * 1e6, "k", lw=0.7)
plt.xlabel("Time after origin (s)")
plt.ylabel("Displacement (µm)")
plt.title("Rayleigh-wave window (15–25 s period, vertical)")
plt.axvline((dist_m / 4200.0), color="green", ls="--", label="4.2 km/s")
plt.axvline((dist_m / 3000.0), color="orange", ls="--", label="3.0 km/s")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Measure A/T in the Rayleigh window
data_R = tr_R.data * 1e6  # µm
dt_R   = tr_R.stats.delta

idx_max_R = np.argmax(data_R)
idx_min_R = np.argmin(data_R)
if idx_max_R < idx_min_R:
    A_pp_R = data_R[idx_max_R] - data_R[idx_min_R]
    T_R = 2.0 * abs(idx_min_R - idx_max_R) * dt_R
else:
    A_pp_R = abs(data_R[idx_min_R] - data_R[idx_max_R])
    T_R = 2.0 * abs(idx_max_R - idx_min_R) * dt_R

A_R = A_pp_R / 2.0

print(f"Rayleigh A (zero-to-peak): {A_R:.3f} µm")
print(f"Rayleigh T: {T_R:.2f} s")

# Prague formula
Ms = np.log10(A_R / T_R) + 1.66 * np.log10(dist_deg) + 3.30
print(f"\n==> Ms = {Ms:.2f}")

---
## Exercise D — Comparison

Let us compare our three measurements with the catalog value.

In [ ]:
labels = ["$M_L$", "$m_b$", "$M_S$", "Catalog\n$M_w$"]
values = [ML, mb, Ms, catalog_mag]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, values, color=colors, edgecolor="k", width=0.5)
ax.axhline(catalog_mag, color="#d62728", ls="--", lw=1, alpha=0.6)
ax.set_ylabel("Magnitude")
ax.set_title("Magnitude Comparison")
ax.set_ylim(0, max(values) + 1)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.08,
            f"{val:.2f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Scale": ["ML", "mb", "Ms", "Catalog Mw"],
    "Value": [f"{ML:.2f}", f"{mb:.2f}", f"{Ms:.2f}", f"{catalog_mag:.1f}"],
    "Band (Hz)": ["1–10", "0.5–2", "0.04–0.067", "—"],
    "Phase / Window": ["WA horizontal", "P vertical", "Rayleigh vertical", "—"],
})
print(summary.to_string(index=False))

---
## Wrap-Up Questions

1. Which of your three estimates is **closest** to the catalog $M_w$?  Why might the others differ?
2. At what magnitude range would you expect $m_b$ to **saturate**?  What about $M_S$?
3. If this earthquake had been at **600 km depth**, which magnitude(s) could you still measure reliably?
4. Richter's $M_L$ was designed for Southern California.  What would change if you applied it to a different tectonic region?

---

*End of Lab 7d*